# 📚 Webアプリで利用するデータ(.csv)を作ります

以下のPDF2枚をご用意ください
1. 「履修要項_[学科名].pdf」：ダウンロードされたフォルダ内にある、ご自身が所属する学科名が記載されたファイル。履修可能な科目一覧を抽出します。
2. 成績.pdf：履修済み授業の確認と、単位の取得状況を確認します。[3S](https://3s.musashi.ac.jp/uprx/up/pk/pky001/Pky00101.xhtml)から、「成績」タブ→「成績照会」→右上の「PDF」でダウンロードされます。成績PDFをアップロードしたくない場合は、ステップ3をスキップしてください。

**使い方（順番に実行してください）**

1. ▶ **ステップ1** を実行 → ライブラリをインストール
2. ▶ **ステップ2** を実行 → 履修要項PDFをアップロード → 科目一覧を抽出
3. ▶ **ステップ3** を実行 → 成績PDFをアップロード → 履修済み・単位状況を抽出
4. ▶ **ステップ4** を実行 → `merged_courses.csv` を生成してダウンロード

> **実行方法：** 各セルの左側にある ▶ ボタンをクリックするか、セルを選択して `Shift + Enter` を押してください。

In [1]:
#@title ▶ ステップ1：準備（最初に1回だけ実行）

print("⏳ ライブラリをインストール中...")
!pip install pdfplumber -q
print("✅ 準備完了！ステップ2に進んでください。")

⏳ ライブラリをインストール中...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 983.4 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 31.4 MB/s eta 0:00:00
✅ 準備完了！ステップ2に進んでください。


In [2]:
#@title ▶ ステップ2：「履修要項_[学科名].pdf」をアップロード → 科目一覧を抽出

import re
import pdfplumber
import pandas as pd
from google.colab import files

# ── アップロード ───────────────────────────────────────────

print("📂 履修要項のPDFファイルを選択してください")
print()
uploaded = files.upload()

if not uploaded:
    print("❌ ファイルが選択されませんでした。もう一度実行してください。")
else:
    syllabus_pdf = list(uploaded.keys())[0]
    print(f"\n✅ アップロード完了：{syllabus_pdf}")
    print("\n⏳ 科目を抽出中...")

    # ── ヘルパー ───────────────────────────────────────────

    def clean(s):
        if s is None:
            return ""
        return re.sub(r"\(cid:\d+\)", "", str(s)).replace("\n", " ").strip()

    def is_ditto(s):
        s = s.strip()
        return bool(s) and (len(s) == 1 and ord(s) == 0x3003 or s in {"//", "\u2033", "\u02ba"})

    NUMBERING  = re.compile(r"^[A-Z]{2,4}\d{4,5}")
    H1         = re.compile(r"^\d+[．.]\s*(.+)$")
    H2         = re.compile(r"^[［\[](.+)[］\]]$")
    H3         = re.compile(r"^[◇◆](.+)$")
    IGNORE_H1  = {"メディア社会学科 全科目一覧"}

    # ── 抽出 ───────────────────────────────────────────────

    num_to_ctx = {}
    cur = {"h1": "", "h2": "", "h3": "", "req": ""}

    with pdfplumber.open(syllabus_pdf) as pdf:

        # Pass 1: テキストスキャンで見出しコンテキストを収集
        for page in pdf.pages:
            text = page.extract_text() or ""
            for line in text.split("\n"):
                lc = re.sub(r"\(cid:\d+\)", "", line).strip()
                if m := H1.match(lc):
                    val = m.group(1).strip()
                    if val not in IGNORE_H1:
                        cur = {"h1": val, "h2": "", "h3": "", "req": ""}
                elif m := H2.match(lc):
                    cur["h2"] = m.group(1).strip()
                    cur["h3"] = ""
                    cur["req"] = ""
                elif m := H3.match(lc):
                    val = re.sub(r"\s*章\s*$", "", m.group(1)).strip()
                    if "必修科目" in val:
                        cur["req"] = "必修"
                    elif "選択科目" in val:
                        cur["req"] = "選択"
                    else:
                        cur["h3"] = val
                tok = lc.split()[0] if lc.split() else ""
                if NUMBERING.match(tok) and tok not in num_to_ctx:
                    num_to_ctx[tok] = dict(cur)

        # Pass 2: テーブルスキャンで科目行を抽出
        course_rows = []
        seen      = set()
        last_num  = ""
        last_ctx  = {"h1": "", "h2": "", "h3": "", "req": ""}

        for page in pdf.pages:
            for table in page.extract_tables():
                for row in table:
                    cells = [clean(c) for c in row]
                    if not cells:
                        continue
                    first = cells[0].strip() if cells[0] else ""

                    if NUMBERING.match(first):
                        num, offset = first, 1
                        last_num = num
                    elif is_ditto(first) and last_num:
                        num, offset = last_num, 1
                    else:
                        found = False
                        for ci, cell in enumerate(cells):
                            if cell and NUMBERING.match(cell.strip()):
                                num, offset = cell.strip(), ci + 1
                                last_num = num
                                found = True
                                break
                        if not found:
                            continue

                    rest = cells[offset:]
                    if len(rest) < 4:
                        continue
                    name = rest[0].strip() if rest[0] else ""
                    if not name:
                        continue

                    key = (num, name)
                    if key in seen:
                        continue
                    seen.add(key)

                    ctx = num_to_ctx.get(num, last_ctx)
                    last_ctx = ctx
                    course_rows.append({
                        "ナンバリング": num,
                        "科目名":       name,
                        "単位":         rest[1].strip() if len(rest) > 1 else "",
                        "配当年次":     (rest[2].strip() if len(rest) > 2 else "").replace("〜", "-").replace("～", "-"),
                        "授業形態":     rest[3].strip() if len(rest) > 3 else "",
                        "備考":         rest[4].strip() if len(rest) > 4 else "",
                        "大分類":       ctx["h1"],
                        "中分類":       ctx["h2"],
                        "小分類":       ctx["h3"],
                        "必修選択":     ctx["req"],
                    })

    # ── 結果表示 ────────────────────────────────────────────

    df_courses = pd.DataFrame(course_rows)
    summary = df_courses.groupby(["大分類", "中分類", "小分類", "必修選択"]).size().reset_index(name="件数")

    print(f"\n✅ 抽出完了：{len(df_courses)} 件")
    print()
    print("【分類サマリー】")
    print(summary.to_string(index=False))
    print("\nステップ3に進んでください。")

📂 履修要項のPDFファイルを選択してください



Saving 履修要項_社会学部_科目.pdf to 履修要項_社会学部_科目.pdf

✅ アップロード完了：履修要項_社会学部_科目.pdf

⏳ 科目を抽出中...

✅ 抽出完了：662 件

【分類サマリー】
  大分類      中分類         小分類 必修選択  件数
外国語科目    必修外国語                    6
外国語科目    選択外国語     全学共通クラス       76
外国語科目    選択外国語   社会学部専用クラス       10
 専門科目     ゼミ科目               必修   8
 専門科目 全学対象専門科目    人文学部提供科目      143
 専門科目 全学対象専門科目  国際教養学部提供科目       68
 専門科目 全学対象専門科目   学芸員課程関連科目       12
 専門科目 全学対象専門科目    教職課程関連科目        8
 専門科目 全学対象専門科目 留学・国際交流関連科目       61
 専門科目 全学対象専門科目    経済学部提供科目       57
 専門科目   学部共通科目     他学科専門科目       37
 専門科目   学部共通科目      社会学部特講        5
 専門科目     展開科目               選択  33
 専門科目     方法科目               必修   3
 専門科目     方法科目               選択  20
 専門科目     理論科目               必修   2
 専門科目     理論科目               選択   3
 総合科目                           110

ステップ3に進んでください。


In [7]:
#@title ▶ ステップ3：成績PDFをアップロード → 履修済み・単位状況を抽出（スキップ可。何もせずステップ4に行くか、「Cancel Upload」を選択）

import pdfplumber
from google.colab import files

# ── アップロード ───────────────────────────────────────────

print("📂 成績一覧表のPDFファイルを選択してください")
print()
uploaded = files.upload()

if not uploaded:
    print("❌ 成績PDFのアップロードがキャンセルされました。")
else:
    grades_pdf = list(uploaded.keys())[0]
    print(f"\n✅ アップロード完了：{grades_pdf}")
    print("\n⏳ 成績データを抽出中...")

    # ── ヘルパー ───────────────────────────────────────────

    def norm_width(s):
        """全角英数字・スラッシュを半角に正規化"""
        if not s:
            return ""
        result = []
        for c in s:
            code = ord(c)
            if 0xFF21 <= code <= 0xFF3A:
                result.append(chr(code - 0xFF21 + ord("A")))
            elif 0xFF41 <= code <= 0xFF5A:
                result.append(chr(code - 0xFF41 + ord("a")))
            elif 0xFF10 <= code <= 0xFF19:
                result.append(chr(code - 0xFF10 + ord("0")))
            elif c == "／":
                result.append("/")
            else:
                result.append(c)
        return "".join(result).strip()

    def to_int(s):
        try:
            return int(norm_width(s or "").strip())
        except:
            return ""

    def is_course_row(cells):
        if len(cells) < 5:
            return False
        return norm_width(cells[3]).isdigit() and len(norm_width(cells[3])) == 4 \
               and "学期" in norm_width(cells[4])

    FAILED_GRADES = {"D", "/"}

    # ── 履修済み科目の抽出（p.1〜3）────────────────────────

    completed_names = set()
    danger_names    = set()
    grade_list      = []

    with pdfplumber.open(grades_pdf) as pdf:
        for page in pdf.pages[:3]:
            tables = page.extract_tables()
            if not tables:
                continue
            for table in tables:
                for row in table:
                    if not row:
                        continue
                    cells = [norm_width(c) for c in row]
                    if not is_course_row(cells):
                        continue
                    name  = cells[0]
                    grade = cells[2]
                    grade_list.append({"科目名": name, "評価": grade})
                    key = name.strip()
                    if grade in FAILED_GRADES:
                        danger_names.add(key)
                    else:
                        completed_names.add(key)

    taken  = [g for g in grade_list if g["評価"] not in FAILED_GRADES]
    failed = [g for g in grade_list if g["評価"] in FAILED_GRADES]

    print(f"   単位取得済み：{len(taken)} 件")
    if failed:
        print(f"   未取得（D・/）：{len(failed)} 件")
        for g in failed:
            print(f"     ⚠️  {g['科目名']}（評価: {g['評価']}）")

    # ── 単位修得状況の抽出（単位修得状況表ページ）──────────

    print("\n⏳ 単位修得状況を抽出中...")

    credits_data = []

    with pdfplumber.open(grades_pdf) as pdf:
        target = next(
            (pg for pg in pdf.pages if "単位修得状況" in (pg.extract_text() or "")),
            None
        )

    if target is None:
        print("   ⚠️  単位修得状況表が見つかりませんでした（スキップ）")
    else:
        parent_level = 0
        seen_total   = False

        for table in target.extract_tables():
            for row in table:
                if not row:
                    continue
                cells = [norm_width(c or "").strip() for c in row]
                name  = cells[0] if cells else ""
                if not name or name == "科目分類" or "履修合計" in name:
                    continue

                if name == "合計":
                    if seen_total:
                        continue
                    seen_total   = True
                    level        = 0
                    parent_level = 0
                elif name.startswith("◆"):
                    level        = 1
                    name         = name[1:].strip()
                    parent_level = 1
                elif name.startswith("◇"):
                    level        = 2
                    name         = name[1:].strip()
                    parent_level = 2
                else:
                    level = 2 if parent_level <= 1 else 3

                credits_data.append({
                    "name":     name,
                    "level":    level,
                    "required": to_int(cells[1]) if len(cells) > 1 else "",
                    "earned":   to_int(cells[2]) if len(cells) > 2 else "",
                    "enrolled": to_int(cells[3]) if len(cells) > 3 else "",
                    "total":    to_int(cells[4]) if len(cells) > 4 else "",
                })

        INDENT = ["", "  ", "    ", "      "]
        for r in credits_data:
            pad  = INDENT[min(r["level"], 3)]
            req  = str(r["required"]) if r["required"] != "" else "─"
            earn = str(r["earned"])   if r["earned"]   != "" else "─"
            print(f"   {pad}{r['name']:<18}  要件 {req:>3}  取得 {earn:>3}")

    print("\n✅ 抽出完了！ステップ4に進んでください。")

📂 成績一覧表のPDFファイルを選択してください



❌ 成績PDFのアップロードがキャンセルされました。


In [ ]:
#@title ▶ ステップ4：merged_courses.csv を生成してダウンロード

import csv
import io
from google.colab import files

# ── 事前チェック ───────────────────────────────────────────

errors = []
if "course_rows" not in dir() or not course_rows:
    errors.append("ステップ2（履修要項PDF）が完了していません")

if errors:
    for e in errors:
        print(f"❌ {e}")
    raise SystemExit

# ── ステップ3 スキップ検知 ────────────────────────────────

skipped_step3 = "completed_names" not in dir()

if skipped_step3:
    completed_names = set()
    danger_names    = set()
    credits_data    = None
    print("⚠️  ステップ3がスキップされました")
    print("   → 履修済みマーク・単位状況はCSVに含まれません")
    print()

print("⏳ マージ中...")

# ── ヘルパー ───────────────────────────────────────────────

STRIP_SUFFIXES = [
    "（総合）", "（会話）", "（文系）", "（理系）",
    "［春学期］", "［秋学期］", "［春秋学期］", "［集中講義］",
]

def normalize_name(name):
    if not name:
        return ""
    s = name.replace("\u3000", " ")
    for suffix in STRIP_SUFFIXES:
        s = s.replace(suffix, "")
    return s.strip()

# ── フラグ付与 ─────────────────────────────────────────────

output_rows    = []
matched        = []
matched_danger = []
unmatched      = list(completed_names | danger_names)
fieldnames     = list(course_rows[0].keys()) + ["履修済み", "危険"]

for row in course_rows:
    key          = normalize_name(row.get("科目名", ""))
    is_completed = key in completed_names
    is_danger    = key in danger_names
    row["履修済み"] = "True" if is_completed else "False"
    row["危険"]     = "True" if is_danger    else "False"
    if is_completed:
        matched.append(row["科目名"])
        if key in unmatched: unmatched.remove(key)
    if is_danger:
        matched_danger.append(row["科目名"])
        if key in unmatched: unmatched.remove(key)
    output_rows.append(row)

# ── CSV出力（メモリ上で組み立て）──────────────────────────

buf = io.StringIO()
writer = csv.DictWriter(buf, fieldnames=fieldnames)
writer.writeheader()
writer.writerows(output_rows)

if credits_data:
    buf.write("\n##CREDITS##\n")
    cred_writer = csv.DictWriter(buf, fieldnames=["name", "level", "required", "earned", "enrolled", "total"])
    cred_writer.writeheader()
    cred_writer.writerows(credits_data)

csv_text = buf.getvalue()

output_filename = "merged_courses_no-personal-data.csv" if skipped_step3 else "merged_courses.csv"

with open(output_filename, "w", encoding="utf-8-sig", newline="") as f:
    f.write(csv_text)

# ── 結果サマリー ──────────────────────────────────────────

print()
print("=" * 40)
print("✅ マージ完了")
print("=" * 40)
print(f"  科目一覧の総科目数  ：{len(output_rows)} 科目")
print(f"  履修済みマッチ      ：{len(matched)} 件")
print(f"  危険マッチ          ：{len(matched_danger)} 件")
print(f"  単位修得状況        ：{'✅ 追記済み' if credits_data else '─ なし'}")

if matched:
    print()
    print("【履修済み】")
    for name in matched:
        print(f"  ✅ {name}")

if matched_danger:
    print()
    print("【未取得（D・/）】")
    for name in matched_danger:
        print(f"  ⚠️  {name}")

if unmatched:
    print()
    print("【成績にあるが科目一覧で見つからなかった科目】")
    for name in sorted(unmatched):
        print(f"  ❓ {name}")

# ── ダウンロード ───────────────────────────────────────────

print()
print(f"⬇️  ダウンロード中: {output_filename}")
files.download(output_filename)
print("✅ ダウンロード完了！")
print()
print("📌 次のステップ")
print("   武蔵履修プランナー（rishu_planner.html）を開き、")
print(f"   {output_filename} を読み込んでください。")